# Online Retail - Exploratory Data Analysis

**Dataset:** UCI Online Retail dataset - transactional data for a UK-based online gift retailer, covering **2010-12-01 to 2011-12-09**.

Each row is one product line within an invoice: `InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`.

This notebook first inspects the raw data, then defines the business questions the analysis will answer.

## 1. Load and Inspect the Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_excel('Online Retail.xlsx')
print('Shape:', df.shape)
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


### Column overview

| Column | Type | Meaning |
|---|---|---|
| InvoiceNo | object | Invoice/transaction number; prefixed `C` = a cancellation |
| StockCode | object | Product/item code |
| Description | object | Product name |
| Quantity | int64 | Units purchased (negative = return/cancellation) |
| InvoiceDate | datetime64 | Date and time of the transaction |
| UnitPrice | float64 | Price per unit, in GBP |
| CustomerID | float64 | Unique customer identifier (many missing) |
| Country | object | Customer's country |

## 2. Data Quality Checks

In [3]:
# Missing values
print('Missing values per column:')
print(df.isna().sum())
print()
print(f"Missing CustomerID: {df['CustomerID'].isna().mean():.1%}")
print(f"Missing Description: {df['Description'].isna().mean():.1%}")

Missing values per column:


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Missing CustomerID: 24.9%
Missing Description: 0.3%


In [4]:
# Duplicate rows
print('Fully duplicated rows:', df.duplicated().sum())

# Numeric ranges
print()
print('Quantity describe:')
print(df['Quantity'].describe())
print()
print('UnitPrice describe:')
print(df['UnitPrice'].describe())

Fully duplicated rows: 5268

Quantity describe:
count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

UnitPrice describe:
count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


In [5]:
# Cancellations and invalid prices
is_cancelled = df['InvoiceNo'].astype(str).str.startswith('C')
print('Cancelled invoice lines (InvoiceNo starts with C):', is_cancelled.sum())
print('Negative Quantity rows:', (df['Quantity'] < 0).sum())
print('Zero/negative UnitPrice rows:', (df['UnitPrice'] <= 0).sum())
print()
print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
print()
print('Unique customers:', df['CustomerID'].nunique())
print('Unique products (StockCode):', df['StockCode'].nunique())
print('Unique invoices:', df['InvoiceNo'].nunique())
print('Unique countries:', df['Country'].nunique())

Cancelled invoice lines (InvoiceNo starts with C): 9288
Negative Quantity rows: 10624
Zero/negative UnitPrice rows: 2517

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00

Unique customers: 4372
Unique products (StockCode): 4070
Unique invoices: 25900
Unique countries: 38


In [6]:
# Country distribution
df['Country'].value_counts().head(15)

Country
United Kingdom     495478
Germany              9495
France               8557
EIRE                 8196
Spain                2533
Netherlands          2371
Belgium              2069
Switzerland          2002
Portugal             1519
Australia            1259
Norway               1086
Italy                 803
Channel Islands       758
Finland               695
Cyprus                622
Name: count, dtype: int64

### Inspection findings

- **541,909 rows x 8 columns**, spanning 2010-12-01 to 2011-12-09 (about one year).
- **CustomerID missing in ~25%** of rows -> those rows can't be used for customer-level analysis (RFM, retention, CLV).
- **Description missing** in a small number of rows, usually paired with `UnitPrice = 0` and no CustomerID -> likely bad/adjustment entries, not real sales.
- **~5,300 fully duplicated rows** -> likely double-scanned line items, should be dropped before aggregation.
- **Negative Quantity** (~10.6k rows) mostly corresponds to cancellations (`InvoiceNo` starting with `C`) -> returns/cancellations, not sales.
- **Zero or negative UnitPrice** (~2.5k rows) -> manual adjustments, bank charges, write-offs rather than product sales.
- **91% of transactions are from the United Kingdom**, with Germany, France and EIRE as the next largest markets -> heavily UK-concentrated business with a long tail of international customers.
- **4,372 unique customers, 4,070 unique products, 25,900 unique invoices.**

These findings define the cleaning steps needed (drop duplicates, separate cancellations from sales, handle missing CustomerID/Description, filter non-product StockCodes) before deeper analysis.

## 3. Business Questions

Based on the columns available (invoice, product, quantity, price, date, customer, country) and the data quality issues found above, this analysis will answer:

1. **Revenue trends** - How does revenue evolve month over month across the observed year? Are there seasonal peaks (e.g. pre-Christmas)?
2. **Top products** - Which products (by `StockCode`/`Description`) generate the most revenue and the most units sold? Are the best-sellers by volume the same as the best-sellers by revenue?
3. **Geographic breakdown** - How much revenue and how many orders come from the UK vs. international markets? Which non-UK countries matter most?
4. **Customer value (RFM)** - Among customers with a known `CustomerID`, who are the highest-value customers by Recency, Frequency and Monetary value? What share of revenue comes from the top 20% of customers?
5. **Customer retention** - What proportion of customers are one-time buyers vs. repeat buyers, and how does repeat-purchase rate change month to month (cohort view)?
6. **Order economics** - What is the distribution of order (invoice) value and basket size (items per invoice)? Are there outlier bulk orders that skew averages?
7. **Returns/cancellations** - What share of transactions are cancellations, which products are cancelled most, and how much revenue do they offset?
8. **Purchase timing** - Are there patterns by day of week or time of day that could inform staffing, marketing send times, or promotions?

Each question will get its own section later in the notebook, built on a cleaned version of `df` (duplicates dropped, cancellations separated out, invalid price/quantity rows handled).

## 4. Data Cleaning

Turning the inspection findings above into an actual cleaned dataset using Pandas: fix data types, drop duplicates, and separate cancelled orders from real sales.

### 4.1 Fix data types

`CustomerID` loads as `float64` only because missing values force pandas off `int64`. Pandas' nullable `Int64` type keeps it as an integer while still allowing NA. `InvoiceNo` and `StockCode` are kept as strings (`object`) since they contain non-numeric codes (e.g. `C536379`, `POST`).

In [ ]:
df_clean = df.copy()

# CustomerID: float64 -> nullable Int64 (keeps NA support, but as an integer id)
df_clean['CustomerID'] = df_clean['CustomerID'].astype('Int64')

# Strip stray whitespace from string/code columns
for col in ['InvoiceNo', 'StockCode', 'Description', 'Country']:
    df_clean[col] = df_clean[col].astype('string').str.strip()

df_clean.dtypes

### 4.2 Remove duplicate rows

`duplicated()` flags rows that are identical across every column - almost certainly the same line item scanned twice, not two separate purchases.

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

print(f'Dropped {before - after:,} duplicate rows ({(before - after) / before:.1%})')
print('Remaining rows:', after)

### 4.3 Flag cancelled orders

Cancellations are identified by `InvoiceNo` starting with `C` (Pandas' `.str.startswith`), which lines up almost exactly with negative `Quantity`. A boolean `IsCancelled` column keeps cancellations in the dataframe (so they can still be analyzed) while making it trivial to filter them out for revenue calculations.

In [ ]:
df_clean['IsCancelled'] = df_clean['InvoiceNo'].str.startswith('C')

n_cancelled = df_clean['IsCancelled'].sum()
print(f"Cancelled order lines: {n_cancelled:,} ({n_cancelled / len(df_clean):.1%} of rows)")

# Cross-check against Quantity sign: cancellations should be (almost) all negative-quantity rows
crosstab = pd.crosstab(df_clean['IsCancelled'], df_clean['Quantity'] < 0)
crosstab.columns = ['Quantity >= 0', 'Quantity < 0']
crosstab.index = ['Not cancelled', 'Cancelled']
crosstab

### 4.4 Re-check missing values after cleaning

`CustomerID` and `Description` are left as missing rather than dropped or imputed - they're genuine gaps in the source data (guest-style checkouts, manual adjustment entries), and different business questions will handle them differently (e.g. product analysis can ignore missing `CustomerID`, customer analysis must filter it out).

In [ ]:
print('Missing values after cleaning:')
print(df_clean.isna().sum())
print()
print('Final dtypes:')
print(df_clean.dtypes)
print()
print('Final shape:', df_clean.shape)

### Cleaning summary

- Converted `CustomerID` to nullable `Int64` and stripped whitespace from string/code columns.
- Dropped ~5,268 fully duplicated rows.
- Added an `IsCancelled` flag (`InvoiceNo` starts with `C`), confirmed it aligns almost exactly with negative `Quantity`.
- Left missing `CustomerID`/`Description` in place - to be filtered per-analysis rather than dropped globally.

`df_clean` is now the working dataframe for the business-question sections that follow.